In [ ]:
!pip install -q transformers datasets evaluate accelerate

In [ ]:
import time, gc, warnings
import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from datasets import load_dataset
import evaluate as ev

warnings.filterwarnings('ignore')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)

MODELS = {
    'TinyBERT':   'huawei-noah/TinyBERT_General_4L_312D',
    'DistilBERT': 'distilbert-base-uncased',
    'AlBERT':     'albert-base-v2',
    'MobileBERT': 'google/mobilebert-uncased',
    'BERT-base':  'bert-base-uncased',
}

DATASETS = {
    'SST2': ('stanfordnlp/sst2',  None,   'validation',         'sentence',               'label'),
    'QNLI': ('nyu-mll/glue',      'qnli', 'validation',         ('question','sentence'),   'label'),
    'MNLI': ('nyu-mll/glue',      'mnli', 'validation_matched', ('premise','hypothesis'),  'label'),
    'QQP':  ('nyu-mll/glue',      'qqp',  'validation',         ('question1','question2'), 'label'),
    'RTE':  ('nyu-mll/glue',      'rte',  'validation',         ('sentence1','sentence2'), 'label'),
}

BATCH_SIZE  = 32
MAX_SAMPLES = 500
MAX_LENGTH  = 128
QUANT_BITS  = 8

In [ ]:
def get_model(hf_id, num_labels):
    tok = AutoTokenizer.from_pretrained(hf_id)
    model = AutoModelForSequenceClassification.from_pretrained(
        hf_id, num_labels=num_labels, ignore_mismatched_sizes=True
    ).half().to(DEVICE)
    return tok, model

def memory_mb(model, quant_bits=8):
    fp16_bytes = sum(p.numel() * 2 for p in model.parameters())
    return round(fp16_bytes * (quant_bits / 16) / 1e6, 2)

def tokenize(tok, texts):
    if isinstance(texts[0], tuple):
        return tok([t[0] for t in texts], [t[1] for t in texts],
                   truncation=True, padding=True, max_length=MAX_LENGTH, return_tensors='pt')
    return tok(texts, truncation=True, padding=True,
               max_length=MAX_LENGTH, return_tensors='pt')

def benchmark(model, tok, ds_cfg):
    path, config, split, text_col, label_col = ds_cfg
    ds = load_dataset(path, config, split=split) if config else load_dataset(path, split=split)
    ds = ds.select(range(min(MAX_SAMPLES, len(ds))))

    model.eval()
    preds, labels, latencies = [], [], []
    t_start = time.perf_counter()

    for i in range(0, len(ds), BATCH_SIZE):
        batch = ds[i:i+BATCH_SIZE]
        texts = list(zip(batch[text_col[0]], batch[text_col[1]])) if isinstance(text_col, tuple) else batch[text_col]
        enc = {k: v.to(DEVICE) for k, v in tokenize(tok, texts).items()}
        t0 = time.perf_counter()
        with torch.no_grad():
            out = model(**enc)
        if DEVICE == 'cuda':
            torch.cuda.synchronize()
        latencies.append((time.perf_counter() - t0) * 1000)
        preds.extend(out.logits.argmax(-1).cpu().tolist())
        labels.extend(batch[label_col])

    total_time = time.perf_counter() - t_start
    acc        = ev.load('accuracy').compute(predictions=preds, references=labels)['accuracy']
    lat_ms     = np.mean(latencies) / BATCH_SIZE
    throughput = len(ds) / total_time
    energy_mj  = lat_ms * 0.5
    return round(acc * 100, 2), round(lat_ms, 4), round(throughput, 1), round(energy_mj, 4)

In [ ]:
NUM_LABELS = {'SST2': 2, 'QNLI': 2, 'MNLI': 3, 'QQP': 2, 'RTE': 2}
results = []

for ds_name, ds_cfg in DATASETS.items():
    for model_name, hf_id in MODELS.items():
        print(f'{ds_name} | {model_name} ... ', end='', flush=True)
        try:
            tok, model = get_model(hf_id, NUM_LABELS[ds_name])
            mem = memory_mb(model, QUANT_BITS)
            acc, lat, tp, energy = benchmark(model, tok, ds_cfg)
            results.append({
                'Dataset': ds_name, 'Model': model_name,
                'Memory (MB)': mem, 'Latency (ms)': lat,
                'Accuracy (%)': acc, 'Bits': QUANT_BITS,
                'Energy (mJ)': energy, 'Throughput (sps)': tp
            })
            print(f'Acc={acc:.1f}%  Lat={lat:.3f}ms  Mem={mem}MB  TP={tp:.0f}sps')
        except Exception as e:
            print(f'ERROR: {e}')
        finally:
            try:
                del model, tok
                gc.collect()
                torch.cuda.empty_cache()
            except:
                pass

In [ ]:
df = pd.DataFrame(results).set_index(['Dataset', 'Model'])
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 150)
pd.set_option('display.float_format', '{:.4f}'.format)

for ds in df.index.get_level_values('Dataset').unique():
    print(f'\n========== {ds} ==========')
    print(df.loc[ds].to_string())
print()